In [1]:
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from matplotlib.patches import Patch
from plotting_defaults import COLOR_MAP, delete_tex_cache, plot_legend, save_and_measure, set_style

from fedaugment.utils import LogParser, cd_to_git_root

cd_to_git_root()
delete_tex_cache()
set_style()

_ = pl.Config.set_tbl_rows(100)

log_dir = Path("logs/curation/webtable")
plot_dir = Path("analysis/plots/curation")
plot_dir.mkdir(exist_ok=True, parents=True)

2026-04-22 17:23:59.256 | DEBUG    | fedaugment.utils.misc:cd_to_git_root:154 - Changed working directory to: /home/lbhm/Projects/fedaugment


In [2]:
parser = LogParser()
logs = parser.parse(log_dir / "train_2026-04-16_16-46-29.log")
logs.extend(parser.parse(log_dir / "train_2026-04-20_11-56-41.log"))
logs.extend(parser.parse(log_dir / "train_2026-04-20_11-57-50.log"))
logs.extend(parser.parse(log_dir / "train_2026-04-20_11-57-58.log"))

In [3]:
data = defaultdict(dict)
summary_rows = []
for log in logs:
    if log["config"]:
        exp_name: str = log["config"]["experiment_name"].replace("webtable_", "")
        exp_config = exp_name.split("-")
        method = exp_config[0]
        if len(exp_config) > 1:
            n_samples = int(exp_config[3][2:])
            sampling_ratio = (
                0.05 if n_samples == 630493 else 0.005 if n_samples == 63049 else 0.001
            )
        else:
            sampling_ratio = 1.0
        with_pca = "pca" in exp_name

        summary_rows.append(
            log["test_metrics"].with_columns(
                pl.lit(exp_name).alias("exp_name"),
                pl.lit(method).alias("method"),
                pl.lit(sampling_ratio).alias("sampling_ratio"),
                pl.lit(with_pca).alias("with_pca"),
                (pl.col("same_lbl_cossim_mean") / pl.col("cossim_mean")).alias(
                    "same_to_all_ratio"
                ),
            )
        )

        if with_pca:
            method += "-pca"
        if "630493" in exp_name:
            data["0.05"][method] = log["test_metrics"]["entity_stability_50"].item()
        elif "63049" in exp_name:
            data["0.005"][method] = log["test_metrics"]["entity_stability_50"].item()
        elif "12609" in exp_name:
            data["0.001"][method] = log["test_metrics"]["entity_stability_50"].item()
        elif "full" in exp_name:
            data["1"]["full"] = log["test_metrics"]["entity_stability_50"].item()

summary = pl.concat(summary_rows).sort("entity_stability_50", descending=True)
summary

loss,cossim_mean,entity_stability_15,entity_stability_5,entity_stability_50,same_lbl_cossim_mean,exp_name,method,sampling_ratio,with_pca,same_to_all_ratio
f64,f64,f64,f64,f64,f64,str,str,f64,bool,f64
2.23452,0.00033,0.69664,0.69081,0.76235,0.98679,"""full""","""full""",1.0,false,2990.272727
2.41166,0.00025,0.60934,0.58213,0.68725,0.95477,"""random-mpnet-dj_adpt-k=630493""","""random""",0.05,false,3819.08
2.89962,0.00043,0.44451,0.41363,0.53886,0.88921,"""random-mpnet-dj_adpt-k=63049""","""random""",0.005,false,2067.930233
2.59451,0.00283,0.40625,0.35596,0.49371,0.94397,"""grid-mpnet-dj_adpt-k=630493-pc…","""grid""",0.05,true,333.558304
2.61271,0.00268,0.40314,0.35207,0.49023,0.94037,"""grid-mpnet-dj_adpt-k=630493""","""grid""",0.05,false,350.884328
3.18723,0.00351,0.28088,0.23529,0.36492,0.85699,"""grid-mpnet-dj_adpt-k=63049""","""grid""",0.005,false,244.156695
3.17159,0.00317,0.27844,0.23342,0.36332,0.8598,"""grid-mpnet-dj_adpt-k=63049-pca…","""grid""",0.005,true,271.230284
3.32579,0.00042,0.24922,0.22525,0.33538,0.85622,"""random-mpnet-dj_adpt-k=12609""","""random""",0.001,false,2038.619048
2.85149,0.0072,0.23793,0.18792,0.33422,0.96006,"""fft_cos-mpnet-dj_adpt-k=630493…","""fft_cos""",0.05,true,133.341667


In [4]:
# Max width: 241.14 pt
fig, ax = plt.subplots(figsize=(3.525, 1.5), layout="tight")

current_x = 0
group_gap = 1
xticks: list[float] = []
xticklabels: list[str] = []

for sampling_rate, methods in sorted(data.items()):
    ordered_methods = dict(sorted(methods.items()))
    positions = [current_x + idx for idx in range(len(ordered_methods))]
    xticks.append(positions[0] + (positions[-1] - positions[0]) / 2)
    xticklabels.append(sampling_rate)

    for pos, (method_name, value) in zip(positions, ordered_methods.items(), strict=True):
        base_method = method_name.replace("-pca", "")
        color = COLOR_MAP[base_method]
        is_pca = method_name.endswith("-pca")
        hatch = "xxx" if is_pca else None

        bar = ax.bar(pos, value, color=color, hatch=hatch, edgecolor="black", linewidth=0.5)
    current_x += len(ordered_methods) + group_gap

legend_handles = [
    Patch(
        facecolor=COLOR_MAP["fft_cos"],
        edgecolor="black",
        label="Farthest-first traversal (cosine similarity)",
    ),
    Patch(
        facecolor=COLOR_MAP["fft_euc"],
        edgecolor="black",
        label="Farthest-first traversal (Euclidean distance)",
    ),
    Patch(facecolor=COLOR_MAP["grid"], edgecolor="black", label="Grid-based sampling"),
    Patch(facecolor=COLOR_MAP["random"], edgecolor="black", label="Random sampling"),
    Patch(facecolor=COLOR_MAP["full"], edgecolor="black", label="Full train split"),
    Patch(facecolor="white", edgecolor="black", hatch="xxxxx", linewidth=0.5, label="PCA variant"),
]

ax.set_xlabel("Sampling rate")
ax.set_ylabel("Entity stability @ 50")
ax.set_xlim(-0.75, current_x - group_gap - 0.25)
ax.set_ylim(0, 1)
ax.set_xticks(xticks, xticklabels)
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.legend(handles=legend_handles, frameon=False, ncol=1)

_ = save_and_measure(fig, plot_dir / "curation.pdf")

analysis/plots/curation/curation.pdf: 239.760x93.959 pt (8.46x3.31 cm)


## In-Depth Analysis

In [5]:
density_analysis_path = Path(log_dir / "curation_density.pq")

In [6]:
density_summary = pl.read_parquet(density_analysis_path).drop(
    ["sample_code", "sample_ratio", "metric", "n_reference", "n_selected"]
)
summary_with_density = summary.join(
    density_summary, how="left", left_on="exp_name", right_on="method"
)

`knn_density_median` acts as a local support proxy: larger values mean the curated samples lie in denser neighborhoods of the reference embedding space. `density_wasserstein` measures how much the subset's density profile shifts away from the full training distribution, while `outlier_rate` tracks how much mass lands in low-support regions. In these runs, large random subsets preserve the highest local density, grid subsets keep very low outlier rates, and FFT-based subsets consistently move toward much sparser regions.

In [7]:
summary_with_density.select(
    "method",
    "sampling_ratio",
    "with_pca",
    "entity_stability_50",
    "knn_distance_mean",
    "knn_distance_median",
    "knn_density_mean",
    "knn_density_median",
    "density_wasserstein",
    "outlier_rate",
).sort("entity_stability_50", descending=True)

method,sampling_ratio,with_pca,entity_stability_50,knn_distance_mean,knn_distance_median,knn_density_mean,knn_density_median,density_wasserstein,outlier_rate
str,f64,bool,f64,f64,f64,f64,f64,f64,f64
"""full""",1.0,false,0.76235,null,null,null,null,null,null
"""random""",0.05,false,0.68725,0.043932,0.017823,2.4654e10,35.809814,1.0463e9,0.099942
"""random""",0.005,false,0.53886,0.043924,0.017503,2.4473e10,36.154854,1.2270e9,0.100033
"""grid""",0.05,true,0.49371,0.067367,0.042862,4758256.5,23.328794,2.5696e10,0.013447
"""grid""",0.05,false,0.49023,0.064671,0.039728,9.51643e6,25.16769,2.5691e10,0.01279
"""grid""",0.005,false,0.36492,0.064919,0.040039,1.5860673e7,24.972935,2.5685e10,0.012514
"""grid""",0.005,true,0.36332,0.067207,0.042899,365.883911,23.309278,2.5701e10,0.012847
"""random""",0.001,false,0.33538,0.044133,0.01755,2.3634e10,35.205059,2.0662e9,0.103339
"""fft_cos""",0.05,true,0.33422,0.158979,0.131801,1.5860616e7,7.586376,2.5685e10,0.078965


In [8]:
# Correlation analysis
target = "entity_stability_50"
rows = []
for metric in [
    "knn_distance_median",
    "knn_distance_mean",
    "knn_density_median",
    "knn_density_mean",
    "density_wasserstein",
    "outlier_rate",
]:
    corr_value = (
        summary_with_density.select([target, metric])
        .drop_nulls()
        .select(pl.corr(target, metric))
        .item()
    )
    rows.append({"metric": metric, "pearson_correlation": corr_value})

correlation_df = pl.DataFrame(rows).sort("pearson_correlation", descending=True)
correlation_df

metric,pearson_correlation
str,f64
"""knn_density_median""",0.816276
"""knn_density_mean""",0.578382
"""density_wasserstein""",-0.578382
"""outlier_rate""",-0.675609
"""knn_distance_mean""",-0.810088
"""knn_distance_median""",-0.814184


### Main Takeaway

Entity stability rises with local density and falls as the curated subset drifts away from the reference corpus distribution. Across the subset metrics, `entity_stability_50` has a strong positive correlation with `knn_density_median`, and negative correlations with `outlier_rate` and `density_wasserstein`. The best curated subsets are therefore not just label-consistent; they also remain concentrated in well-supported parts of the embedding space, whereas the FFT variants produce both sparse and outlier-heavy subsets.

In [9]:
scatter_df = summary_with_density.drop_nulls()

for col, xlabel, log_x in [
    ("knn_distance_median", "Median kNN distance", True),
    ("density_wasserstein", "Wasserstein distance to candidate corpus", True),
    ("outlier_rate", "Outlier rate", False),
]:
    fig, ax = plt.subplots(figsize=(2.3, 1.7), layout="tight")
    sns.scatterplot(
        scatter_df,
        x=col,
        y="entity_stability_50",
        hue="method",
        size="sampling_ratio",
        style="with_pca",
        hue_order=["fft_cos", "fft_euc", "grid", "random"],
        sizes=[10, 20, 30],
        legend=False,
        ax=ax,
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Entity stability @ 50")
    ax.set_ylim(0, None)
    if log_x:
        ax.set_xscale("log")
    else:
        ax.set_xlim(0, None)

    correlation = correlation_df.filter(pl.col("metric") == col)["pearson_correlation"].item()
    ax.set_title(f"Correlation: {correlation:.2f}")

    _ = save_and_measure(fig, plot_dir / f"analysis-{col}.pdf")

# Create dummy plot and extract legend
fig, ax = plt.subplots()
sns.scatterplot(
    scatter_df,
    x=col,
    y="entity_stability_50",
    hue="method",
    size="sampling_ratio",
    style="with_pca",
    hue_order=["fft_cos", "fft_euc", "grid", "random"],
    sizes=[10, 20, 30],
    legend=True,
    ax=ax,
)
handles, labels = ax.get_legend_handles_labels()
handles = handles[1:5] + handles[6:9] + handles[10:12]
labels = [
    "FFT (cosine)",
    "FFT (Euclidean)",
    "Grid sampling",
    "Random sampling",
    "0.001",
    "0.005",
    "0.05",
    "w/o PCA",
    "w/ PCA",
]
plt.close(fig)

plot_legend(plot_dir / "analysis-legend.pdf", handles, labels)

analysis/plots/curation/analysis-knn_distance_median.pdf: 151.560x108.360 pt (5.35x3.82 cm)
analysis/plots/curation/analysis-density_wasserstein.pdf: 151.560x108.360 pt (5.35x3.82 cm)
analysis/plots/curation/analysis-outlier_rate.pdf: 151.560x108.360 pt (5.35x3.82 cm)
analysis/plots/curation/analysis-legend.pdf: 396.257x11.496 pt (13.98x0.41 cm)


## Proxy Model Analysis

In [10]:
parser = LogParser()
proxy_logs = []
for file in log_dir.glob("proxy_analysis_train_*.log"):
    proxy_logs.extend(parser.parse(file))

In [11]:
proxy_rows = []
for log in proxy_logs:
    if log["config"]:
        exp_name: str = log["config"]["experiment_name"].replace("webtable_", "")
        exp_config = exp_name.split("-")
        method = exp_config[0]
        proxy_model = exp_config[1]
        if len(exp_config) > 1:
            n_samples = int(exp_config[3][2:].split("_")[0])
            sampling_ratio = (
                0.05 if n_samples == 630493 else 0.005 if n_samples == 63049 else 0.001
            )
        else:
            sampling_ratio = 1.0
        with_pca = "pca" in exp_name

        proxy_rows.append(
            log["test_metrics"].with_columns(
                pl.lit(exp_name).alias("exp_name"),
                pl.lit(method).alias("method"),
                pl.lit(proxy_model).alias("proxy_model"),
                pl.lit(sampling_ratio).alias("sampling_ratio"),
                pl.lit(with_pca).alias("with_pca"),
                (pl.col("same_lbl_cossim_mean") / pl.col("cossim_mean")).alias(
                    "same_to_all_ratio"
                ),
            )
        )

proxy_data = pl.concat(proxy_rows).sort("entity_stability_50", descending=True)

In [12]:
proxy_summary = (
    proxy_data.filter(pl.col("method") != "random")
    .group_by("method", "with_pca")
    .agg(
        # Entity Stability @ 5
        pl.col("entity_stability_5").mean().alias("es5_mean"),
        pl.col("entity_stability_5").std().alias("es5_std"),
        (pl.col("entity_stability_5").max() - pl.col("entity_stability_5").min()).alias(
            "es5_range"
        ),
        # Entity Stability @ 15
        pl.col("entity_stability_15").mean().alias("es15_mean"),
        pl.col("entity_stability_15").std().alias("es15_std"),
        (pl.col("entity_stability_15").max() - pl.col("entity_stability_15").min()).alias(
            "es15_range"
        ),
        # Entity Stability @ 50
        pl.col("entity_stability_50").mean().alias("es50_mean"),
        pl.col("entity_stability_50").std().alias("es50_std"),
        (pl.col("entity_stability_50").max() - pl.col("entity_stability_50").min()).alias(
            "es50_range"
        ),
        pl.len().alias("n_runs"),
    )
    .sort("es50_mean", descending=True)
)

proxy_summary

method,with_pca,es5_mean,es5_std,es5_range,es15_mean,es15_std,es15_range,es50_mean,es50_std,es50_range,n_runs
str,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""grid""",false,0.229335,0.000253,0.00066,0.27093,0.000323,0.00098,0.361542,0.000402,0.00126,6
"""grid""",true,0.228875,0.000502,0.00152,0.269932,0.000724,0.00206,0.360412,0.000885,0.00246,6
"""fft_cos""",true,0.07754,0.001827,0.00503,0.09283,0.002148,0.00606,0.140555,0.003913,0.01101,6
"""fft_euc""",true,0.053367,0.005733,0.01561,0.05962,0.007514,0.01821,0.08423,0.013098,0.03185,6
"""fft_cos""",false,0.04674,0.005712,0.01591,0.05135,0.006886,0.01804,0.071163,0.011221,0.02711,6
"""fft_euc""",false,0.046642,0.005892,0.01656,0.051183,0.007087,0.01883,0.070927,0.01154,0.02842,6


## Column Name Distribution Analysis

In [13]:
col_name_counter: dict[str, int] = json.loads((log_dir / "col_name_counter.json").read_text())
sorted_counts = sorted(col_name_counter.values(), reverse=True)

In [14]:
for k in [5, 15, 50]:
    cols_above_k = [v for v in col_name_counter.values() if v > k]
    print(f"Column names with more than {k} instances: {len(cols_above_k)}")
    print(
        f"Total number of instances of those columns: "
        f"{sum(cols_above_k)} ({sum(cols_above_k) / sum(col_name_counter.values()):.3f}%)"
    )
    print(sorted(cols_above_k, reverse=True)[:10])

Column names with more than 5 instances: 35424
Total number of instances of those columns: 14510755 (0.977%)
[362772, 244171, 228359, 198378, 192930, 188122, 185286, 166904, 154456, 120833]
Column names with more than 15 instances: 17082
Total number of instances of those columns: 14345328 (0.965%)
[362772, 244171, 228359, 198378, 192930, 188122, 185286, 166904, 154456, 120833]
Column names with more than 50 instances: 8267
Total number of instances of those columns: 14104086 (0.949%)
[362772, 244171, 228359, 198378, 192930, 188122, 185286, 166904, 154456, 120833]


In [15]:
# Ten most common column names
[
    k
    for k, v in col_name_counter.items()
    if v in {362772, 244171, 228359, 198378, 192930, 188122, 185286, 166904, 154456, 120833}
]

['team', 's', 'm', 'name', 'date', 'title', 't', 'w', 'f', 'time']

In [16]:
# Occurences of the 10 most common column names
sum({362772, 244171, 228359, 198378, 192930, 188122, 185286, 166904, 154456, 120833})

2042211